# Statistical inference in Python
**Foundations of Computational Social Science · TU Graz**

In this notebook you work through the lecture in five parts. Wherever you see `___`, replace it with your own code or answer. Cells marked **✎ Your turn** are the ones to complete.

Datasets (both simulated for this course):
- `course_grades.csv`: 40 students of one course, hours studied per week and exam grade (0–100).
- `university_grades.csv`: 20,000 students, with hours studied, attendance (%), age, prior knowledge (0–10) and exam grade.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (6, 4)
RED = "#E4154B"

## Part 1 · Revise the basics

In [ ]:
course = pd.read_csv("data/course_grades.csv")
course.head()

**✎ Your turn.** Compute the mean, standard deviation, variance, minimum and maximum of `exam_grade`.
Check: is the variance the standard deviation squared?

In [ ]:
mean_grade = ___
sd_grade = ___
var_grade = ___
min_grade = ___
max_grade = ___
print(f"mean = {mean_grade:.2f}, sd = {sd_grade:.2f}, variance = {var_grade:.2f}")
print(f"min = {min_grade}, max = {max_grade}")

**✎ Your turn.** Plot a histogram of `exam_grade`. Where are most students? Does any grade stand out?

In [ ]:
plt.hist(___, bins=10, color=RED, edgecolor="white")
plt.axvline(mean_grade, color="black", linestyle="--", label="mean")
plt.xlabel("exam grade (0–100)")
plt.ylabel("number of students")
plt.legend()
plt.show()

## Part 2 · Null hypothesis significance testing

**✎ Your turn.** Before looking at the data, write down your hypotheses and your significance level.

- H0: ___
- H1: ___
- α = ___ (why this value?)

In [ ]:
alpha = ___   # e.g. 0.1, 0.05, 0.01 or 0.001

## Part 3 · Correlation: hours studied and exam grade

### 3a · Scatter plot and correlation
**✎ Your turn.** Draw a scatter plot of `hours_studied` (x) against `exam_grade` (y). Does anything look unusual?

In [ ]:
plt.scatter(___, ___, color=RED)
plt.xlabel("hours studied per week")
plt.ylabel("exam grade (0–100)")
plt.show()

**✎ Your turn.** Compute Pearson's r and its p-value with `stats.pearsonr`. Is the association significant at your α?

In [ ]:
r, p = stats.pearsonr(___, ___)
print(f"r = {r:.2f}, p = {p:.3f}, significant: {p < alpha}")

### 3b · The outlier
**✎ Your turn.** Find the student who doesn't fit the pattern (hint: many hours, very low grade).

In [ ]:
course.sort_values(___).head(3)

Student 40 studied 14.5 hours a week but scored 8 points: they were ill on the exam day.

**✎ Your turn.** Remove this student and compute r and p again. What changes? Is removing them justified, and would you decide this *before* or *after* seeing the result?

In [ ]:
course_clean = course[course["student_id"] != ___]
r2, p2 = stats.pearsonr(___, ___)
print(f"without outlier: r = {r2:.2f}, p = {p2:.6f}")

## Part 4 · Fitting a multiple linear regression

In [ ]:
uni = pd.read_csv("data/university_grades.csv")
uni.describe().round(1)

### 4a · Fit the model
**✎ Your turn.** Fit exam grade on hours studied, attendance and age:

exam_grade = b0 + b1·hours_studied + b2·attendance + b3·age + e

In [ ]:
model = smf.ols("exam_grade ~ ___ + ___ + ___", data=uni).fit()
print(model.summary())

### 4b · Read the output
**✎ Your turn.**
1. Write down the fitted equation.
2. Which predictors are significant at your α?
3. Interpret the coefficient of `hours_studied` in one sentence, starting with "Holding attendance and age constant, …".

### 4c · Model fit and prediction
**✎ Your turn.** What is R², and what does it mean here? (It is in the output above, or `model.rsquared`.)

Then use the model to predict the grade of a 22-year-old who studies 10 hours a week and attends 80% of classes. Then check by plugging the numbers into the equation by hand.

In [ ]:
new_student = pd.DataFrame({"hours_studied": [___], "attendance": [___], "age": [___]})
model.predict(new_student)

In [ ]:
b = model.params
by_hand = b["Intercept"] + b["hours_studied"] * ___ + b["attendance"] * ___ + b["age"] * ___
by_hand

## Part 5 · Issues: significance, sample size and other pitfalls

### 5a · Significant, but meaningful?
**✎ Your turn.** `age` is highly significant. How many points does being 10 years older add to the predicted grade? Is that meaningful?

In [ ]:
model.params["age"] * ___

### 5b · Sample size
**✎ Your turn.** Fit the same model on a random sample of 100 students. Is age still significant? Why does the answer change?

In [ ]:
small_sample = uni.sample(n=___, random_state=1)
model_small = smf.ols("exam_grade ~ hours_studied + attendance + age", data=small_sample).fit()
model_small.pvalues.round(4)

### 5c · An unobserved variable
Students with more prior knowledge also tend to study more. **✎ Your turn.** Add `prior_knowledge` to the model. What happens to the coefficient of `hours_studied`?

In [ ]:
model_prior = smf.ols("exam_grade ~ hours_studied + attendance + age + ___", data=uni).fit()
print(f"hours without prior knowledge: {model.params['hours_studied']:.2f}")
print(f"hours with prior knowledge:    {model_prior.params['hours_studied']:.2f}")

### 5d · Checking assumptions
Residuals against fitted values: for a good linear model the cloud should have no pattern and a constant spread (homoscedasticity).

In [ ]:
plt.scatter(model.fittedvalues, model.resid, s=3, alpha=0.3, color=RED)
plt.axhline(0, color="black")
plt.xlabel("fitted grade")
plt.ylabel("residual")
plt.show()

Q-Q plot of the residuals: if they are roughly normal, the points follow the line.

In [ ]:
import statsmodels.api as sm
sm.qqplot(model.resid, line="s")
plt.show()

Correlation between the predictors (multicollinearity). **✎ Your turn.** Are any predictors very highly correlated (say above 0.8)?

In [ ]:
uni[["hours_studied", "attendance", "age", "prior_knowledge"]].___().round(2)

### 5e · p-hacking in 10 lines
We create 20 variables of pure random noise and test each one against the exam grades of the 40 course students. None of them has any real relation to the grade.

**✎ Your turn.** Run the cell a few times with different seeds. How many "significant" results do you get on average? Why?

In [ ]:
rng = np.random.default_rng(___)
significant = 0
for i in range(20):
    noise = rng.normal(size=len(course))
    r_noise, p_noise = stats.pearsonr(noise, course["exam_grade"])
    if p_noise < 0.05:
        significant += 1
print(f"{significant} of 20 pure-noise variables are 'significant' at p < .05")

**Reflection.** In this notebook you made several decisions: your α, whether to drop student 40, which predictors to include. Which of them could change the conclusion? How could you make them transparent to others?